# 01 — Estimate the backbone: fixed-q vs AQF (single scenario)

**Purpose.** Walk the three estimator families through one scenario, one step at a time, so
the method can be read rather than inferred from `experiment.py`. This is the clearest
statement of how AQF works anywhere in the repository.

**Inputs.**

- `config.SIGMA`, `config.BACKBONE_B`, `config.DAYS`, `config.FIXED_QS`,
  `config.Q_DEFAULT`, `config.D_THRESH`, from `src/config.py`.
- Illustrative parameters set inline: `p = 0.3`, `kappa = 3.0`, `seed = 1`. Chosen so the
  scenario sits just inside the trustworthy region — see the section below.

**Outputs.** None written to disk. Printed inline for inspection.

**Process.**

1. Put `src/` on the path and import `config`, `estimators`, `synth`.
2. Generate one 365-day scenario and keep only the observed load.
3. Estimate the backbone with the **fixed-quantile baseline** at q = 0.1, 0.2, 0.3.
4. Estimate it with **oracle-AQF**, which is allowed to know the true `p` and `kappa`.
5. Estimate it with **estimated-AQF**: fit the mixture, check identifiability, blend.

**The question every step answers.** The true backbone is `config.BACKBONE_B` = 10.0 kW.
Each estimator sees only the 365 load values and must return that number. Watch how close
each one gets, and at what cost in assumptions.

## Setup

`src/` goes on the import path so the notebook uses exactly the same modules as the grid
runner. `estimators` holds all five functions the method is built from.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import config
import estimators
import synth

## The scenario

A full 365 days this time, since we now want a realistic sample size rather than a legible
plot. The two parameters are chosen deliberately:

- **`p = 0.3`** — events on roughly three days in ten. Frequent enough that the mixture has
  something to fit, which is why `config.P_ROBUST` uses the same value as its cutoff.
- **`kappa = 3.0`** — the event is three times the day-to-day noise. This clears
  `config.KAPPA_IDENTIFIABLE` (2.83), so the scenario sits just inside the region the
  method claims to handle. It is deliberately a case that *should* work; the recoverability
  map in notebook 03 is where the failures live.

Only `load` is carried forward. Everything the estimators see from here is that one vector
of 365 numbers — no day labels, no event indicators, no `p` or `kappa`, except in the
oracle step where the leak is the entire point.

In [ ]:
p, kappa = 0.3, 3.0
days_df = synth.generate_days(p=p, kappa=kappa, sigma=config.SIGMA, backbone_b=config.BACKBONE_B, days=config.DAYS, seed=1)
load = days_df["l"].to_numpy()

## Step 1 — The fixed-quantile baseline

This is incumbent practice: pick a low percentile of the across-day distribution and call it
the backbone. The choice of percentile is a judgement call made once and applied everywhere,
which is exactly the habit the paper is arguing with.

Three candidate values are compared because there is no principled way to choose between
them from the data alone — which is the point. Each returns a different backbone, and
nothing in the output tells you which is right. With the true value at 10.0 kW, watch how
far each one falls short: every fixed quantile here sits **below** the truth, because a
quantile chosen without reference to `p` and `kappa` cuts into the wrong part of the
distribution.

In [ ]:
# Fixed-quantile baseline
for q in config.FIXED_QS:
    print(q, estimators.fixed_quantile_backbone(load, q))

## Step 2 — Oracle-AQF

The paper's central relation says that for a backbone estimated as the q-th quantile, the
normalised bias `b` satisfies

$$q = (1-p)\,\Phi(b) + p\,\Phi(b - \kappa)$$

Setting `b = 0` and solving gives the **bias-minimising quantile**:

$$q^{*} = \frac{1-p}{2} + p\,\Phi(-\kappa)$$

That is `estimators.aqf_quantile`, and it is the whole idea: the correct quantile is not a
preference, it is a quantity you can compute from how often events happen and how big they
are.

This step is called the *oracle* because it is handed the true `p` and `kappa` rather than
estimates. No real deployment can do this — it is a reference point showing what the
quantile rule achieves when the inputs are perfect. Note that `q*` lands near 0.35, well
above every value in `config.FIXED_QS`, so the fixed baselines were never going to reach
the right answer at this `(p, kappa)`.

Deliberately **not** called an upper bound: `q*` still takes an empirical quantile of 365
days, so estimated-AQF beats it in parts of the grid.

In [ ]:
# Oracle-AQF (true p, kappa known)
q_star_oracle = estimators.aqf_quantile(p, kappa)
print("q_star_oracle =", q_star_oracle)
print("B_hat_oracle =", estimators.fixed_quantile_backbone(load, q_star_oracle))

## Step 3 — Estimated-AQF: the method as it would actually be deployed

Four moves, in order, using nothing but `load`.

**Fit the mixture.** `fit_mixture` fits a two-component Gaussian mixture with a *tied*
covariance — both components share one `sigma`, which is precisely what `L = B + ZA + eps`
asserts. A `"full"` covariance would be fitting a model the data was not generated from.
The lower-mean component is taken as the non-event state, so `p_hat` is the weight of the
upper component and `kappa_hat` the standardised gap between the means.

**Check identifiability.** `identifiability_diagnostic` returns `kappa_hat / sqrt(2)`. Note
carefully: this is Ashman's D **rescaled by 1/√2**, not Ashman's statistic itself, so
gating at `D_THRESH = 2.0` is the stricter bar `kappa >= 2.83`. The paper says "a
bimodality bar in the spirit of Ashman's D" for this reason and does not claim the
statistic *is* Ashman's D.

**Compute the quantile.** The same `aqf_quantile` as the oracle step, now fed estimates
instead of truth.

**Blend, do not switch.** `fallback_blend` computes `w = clip(D_hat / D_THRESH, 0, 1)` and
returns `w * q_star + (1 - w) * q_default`. Confidence therefore degrades smoothly: a
well-separated mixture is trusted outright, a marginal one is pulled part-way back toward
`config.Q_DEFAULT` (0.2), and a hopeless one falls back to the default entirely. There is
no hard switch — the soft blend is the only implemented path.

**What to look for in the output.** `kappa_hat` comes in a little under the true 3.0, which
drags `d_hat` just below the threshold of 2.0. So the blend weight sits fractionally under
1 and the estimator hedges very slightly toward the default, even though the true `kappa`
clears the bar. That is the gate behaving conservatively, and it is worth seeing once.

In [ ]:
# Estimated-AQF: fit mixture, check identifiability, blend toward the fallback default
fit = estimators.fit_mixture(load, seed=2)
d_hat = estimators.identifiability_diagnostic(fit.kappa_hat)
q_star_hat = estimators.aqf_quantile(fit.p_hat, fit.kappa_hat)
q_final, weight = estimators.fallback_blend(q_star_hat, config.Q_DEFAULT, d_hat, config.D_THRESH)
print(fit)
print("d_hat =", d_hat, " q_star_hat =", q_star_hat, " q_final =", q_final, " weight =", weight)

## Conclusion

Against a true backbone of 10.0 kW, on this scenario:

- The **fixed quantiles** all undershoot, and worsen as q falls — q = 0.1 is the furthest
  out. None of them had a way to know that the right quantile here was about 0.35.
- **Oracle-AQF** lands within a few hundredths of the truth, which is what the closed form
  buys you when `p` and `kappa` are known exactly.
- **Estimated-AQF** lands close behind it, having recovered `p` and `kappa` from the load
  alone. The gap between the two is the price of estimating rather than knowing.

The ordering is the paper's claim in miniature: computing the quantile beats picking one,
and estimating the inputs costs surprisingly little when the mixture is identifiable.

Two honest caveats to carry forward. This scenario was chosen to be inside the trustworthy
region — notebook 03's recoverability map shows where all of this breaks down. And
`fit_mixture` also returns `backbone_hat`, the lower component's mean, which on this
scenario is *closer to the truth than any quantile*; the method discards it deliberately,
trading that efficiency for robustness when the two-Gaussian model does not hold.

Next: notebook 02 runs this same comparison across the full `(p, kappa)` grid.